In [ ]:
import numpy as np
import pandas as pd

In [ ]:
def extract_id(df):
    df.loc[:, "id"] = df.gdrive_id.str.replace(r"(.+)\/d\/(.+)", r"\2", regex=True)
    return df


def sample_cases(df, n_cases=600, random_state=None):
    """
    Randomly sample n_cases from the dataframe, returning all rows for selected cases.
    """
    sampled_cases = (
        df["provisional_case_name"].drop_duplicates().sample(n=n_cases, random_state=random_state)
    )
    return df[df["provisional_case_name"].isin(sampled_cases)].copy()


def read_tbl():
    df = pd.read_csv("../data/output/autofolio_df_concat.csv")
    df = df[
        [
            "provisional_case_name",
            "gdrive_path",
            "gdrive_name",
            "gdrive_id",
            "sha1",
            "source_agency_name",
        ]
    ]
    return df


df = read_tbl()

df = df.pipe(extract_id)

# print(df.head(1))

# print(df.columns)

# print(df.provisional_case_name.nunique())

df_sample = df.pipe(sample_cases).copy()

# print(df_sample.head(1))

# df_sample.to_csv("../data/output/sample.csv", index=False)

df

In [ ]:
df.source_agency_name.unique()

In [ ]:
# THREE-TIER SAMPLING STRATEGY FOR 1,000 NEW CASES
# Tier 1: Targeted sampling from 11 flagged problematic agencies (meet minimum thresholds)
# Tier 2: Random sampling from medical examiner and DA agencies (strata expansion)
# Tier 3: Random sampling from all other agencies (general population)


# ==============================================================================
# SETUP: Define problematic agencies (Tier 1)

# Mapping from original names to standardized names in dataset
agency_name_mapping = {
    "Los Angeles County Medical Examiner-Coroner": "los_angeles_medical_examiner-coroner",
    "Fresno County District Attorney": "fresno_county_district_attorney",
    "Riverside County District Attorney": "riverside_county_district_attorney",
    "Riverside County Sheriff's Department": "riverside_county_sheriff",
    "Oakland Police Department": "oakland_police_department",
    "Los Angeles Police Department": "los_angeles_police_department",
    "California Highway Patrol": "california_highway_patrol",
    "California State Personnel Board": "california_state_personnel_board",
    "California Department of Corrections & Rehabilitation": (
        "california_department_of_corrections_and_rehabilitation"
    ),
    "Los Angeles County Sheriff's Department": "los_angeles_county_sheriff",
    "Los Angeles Civil Service Commission": "los_angeles_county_civil_services_commission",
}

# Get standardized list of Tier 1 agencies
tier_1_agencies = list(agency_name_mapping.values())

# ==============================================================================
# TIER 1: ANALYZE PROBLEMATIC AGENCIES IN FULL DATASET

# Count unique cases per agency in full dataset
full_dataset_counts = df.groupby("source_agency_name")["provisional_case_name"].nunique()

# Create summary table for Tier 1 agencies
tier_1_summary = pd.DataFrame({"total_cases_available": full_dataset_counts}).fillna(0).astype(int)

# Filter to Tier 1 problematic agencies
tier_1_summary = tier_1_summary[tier_1_summary.index.isin(tier_1_agencies)].copy()
tier_1_summary = tier_1_summary.sort_values("total_cases_available", ascending=False)

print("=" * 100)
print("TIER 1: PROBLEMATIC AGENCIES - AVAILABLE CASES")
print("=" * 100)
print(tier_1_summary)
print("\nTier 1 Summary:")
print(f"  - Total agencies: {len(tier_1_agencies)}")
print(f"  - Total cases available: {tier_1_summary['total_cases_available'].sum()}")

# ==============================================================================
# TIER 1: DEFINE ALLOCATION TARGETS

# Set target number of cases to sample from each problematic agency based on size
tier_1_targets = {
    # Large agencies (>500 cases) - sample 100 cases each
    "los_angeles_county_sheriff": 100,
    "los_angeles_police_department": 100,
    # Medium agencies (100-500 cases) - sample 50-60 cases each
    "california_department_of_corrections_and_rehabilitation": 60,
    "oakland_police_department": 50,
    "riverside_county_district_attorney": 50,
    "california_highway_patrol": 50,
    "california_state_personnel_board": 50,
    # Small agencies (<100 cases) - sample 30-40 cases each
    "riverside_county_sheriff": 40,
    "fresno_county_district_attorney": 40,
    "los_angeles_medical_examiner-coroner": 20,  # Limited by 31 total available
    "los_angeles_county_civil_services_commission": 15,  # Limited by 19 total available
}

# Calculate final allocation for each Tier 1 agency
tier_1_allocation = []

for agency, target_sample in tier_1_targets.items():
    available = tier_1_summary.loc[agency, "total_cases_available"]

    # Don't exceed what's actually available
    to_sample = min(target_sample, available)

    tier_1_allocation.append(
        {
            "agency": agency,
            "total_available": available,
            "target_sample": target_sample,
            "actual_sample": to_sample,
            "sampling_rate": round((to_sample / available * 100), 1) if available > 0 else 0,
        }
    )

tier_1_df = pd.DataFrame(tier_1_allocation)
tier_1_df = tier_1_df.sort_values("total_available", ascending=False)

print("\n" + "=" * 100)
print("TIER 1: ALLOCATION PLAN")
print("=" * 100)
print(tier_1_df.to_string(index=False))
print(f"\nTier 1 total cases to sample: {tier_1_df['actual_sample'].sum()}")

# ==============================================================================
# TIER 2: IDENTIFY MEDICAL EXAMINER AND DISTRICT ATTORNEY AGENCIES
# ==============================================================================

# Find all agencies with "medical" in their name (medical examiners, coroners)
medical_examiner_agencies = df[
    df["source_agency_name"].str.contains("medical", case=False, na=False)
]["source_agency_name"].unique()

# Find all agencies with "district_attorney" in their name
district_attorney_agencies = df[
    df["source_agency_name"].str.contains("district_attorney", case=False, na=False)
]["source_agency_name"].unique()

# Combine into single Tier 2 list (remove duplicates)
tier_2_agencies_all = list(set(list(medical_examiner_agencies) + list(district_attorney_agencies)))

# Remove any Tier 2 agencies that are already in Tier 1 (avoid double-counting)
tier_2_agencies = [agency for agency in tier_2_agencies_all if agency not in tier_1_agencies]

print("\n" + "=" * 100)
print("TIER 2: MEDICAL EXAMINER & DISTRICT ATTORNEY AGENCIES")
print("=" * 100)
print(f"Total Tier 2 agencies (after excluding Tier 1 overlaps): {len(tier_2_agencies)}")
print(f"  - Medical Examiner agencies: {len([a for a in tier_2_agencies if 'medical' in a])}")
print(
    f"  - District Attorney agencies: "
    f"{len([a for a in tier_2_agencies if 'district_attorney' in a])}"
)

# ==============================================================================
# TIER 2: CALCULATE CASE AVAILABILITY
# ==============================================================================

# Total unique cases available from Tier 2 agencies in full dataset
tier_2_total_cases = df[df["source_agency_name"].isin(tier_2_agencies)][
    "provisional_case_name"
].nunique()

print("\nTier 2 case availability:")
print(f"  - Total cases available: {tier_2_total_cases}")

# ==============================================================================
# TIER 3: IDENTIFY ALL OTHER AGENCIES (exclude Tier 1 and Tier 2)
# ==============================================================================

# Get all agencies that are NOT in Tier 1 or Tier 2
all_agencies = df["source_agency_name"].unique()
tier_3_agencies = [
    agency
    for agency in all_agencies
    if agency not in tier_1_agencies and agency not in tier_2_agencies
]

# Calculate Tier 3 case availability
tier_3_total_cases = df[df["source_agency_name"].isin(tier_3_agencies)][
    "provisional_case_name"
].nunique()

print("\n" + "=" * 100)
print("TIER 3: ALL OTHER AGENCIES (General Population)")
print("=" * 100)
print(f"Total Tier 3 agencies: {len(tier_3_agencies)}")
print("Tier 3 case availability:")
print(f"  - Total cases available: {tier_3_total_cases}")

# Calculate how much budget remains after Tier 1
tier_1_total = tier_1_df["actual_sample"].sum()
remaining_budget = 1000 - tier_1_total

# Allocate remaining budget between Tier 2 and Tier 3
tier_2_allocation = int(remaining_budget * 0.25)
tier_3_allocation = remaining_budget - tier_2_allocation

print("\n" + "=" * 100)
print("FINAL ALLOCATION PLAN FOR 1,000 NEW CASES")
print("=" * 100)
print(f"Tier 1 (Problematic agencies - targeted):     {tier_1_total:4d} cases")
print(f"Tier 2 (Medical/DA agencies - random):        {tier_2_allocation:4d} cases")
print(f"Tier 3 (All other agencies - random):         {tier_3_allocation:4d} cases")
print(f"{'':48s}{'----':>4s}")
print(f"Total new sample:                              {1000:4d} cases")

print("\n" + "=" * 100)
print("FEASIBILITY CHECK")
print("=" * 100)

feasible = True

if tier_1_total > tier_1_summary["total_cases_available"].sum():
    print(
        f"⚠️  WARNING: Tier 1 requests {tier_1_total} cases but only "
        f"{tier_1_summary['total_cases_available'].sum()} available!"
    )
    feasible = False
else:
    print(
        f"✓ Tier 1: Requesting {tier_1_total} cases, "
        f"{tier_1_summary['total_cases_available'].sum()} available"
    )

if tier_2_allocation > tier_2_total_cases:
    print(
        f"⚠️  WARNING: Tier 2 requests {tier_2_allocation} cases but only "
        f"{tier_2_total_cases} available!"
    )
    feasible = False
else:
    print(f"✓ Tier 2: Requesting {tier_2_allocation} cases, {tier_2_total_cases} available")

if tier_3_allocation > tier_3_total_cases:
    print(
        f"⚠️  WARNING: Tier 3 requests {tier_3_allocation} cases but only "
        f"{tier_3_total_cases} available!"
    )
    feasible = False
else:
    print(f"✓ Tier 3: Requesting {tier_3_allocation} cases, {tier_3_total_cases} available")

if feasible:
    print("\n✓ All tiers are feasible! Allocation plan is ready to execute.")
else:
    print("\n⚠️  Allocation plan needs adjustment based on warnings above.")

print("\n" + "=" * 100)
print("NEXT STEPS")
print("=" * 100)
print("Ready to execute sampling with the following tier definitions:")
print(f"  - tier_1_agencies: {len(tier_1_agencies)} agencies (use tier_1_df for allocation)")
print(f"  - tier_2_agencies: {len(tier_2_agencies)} agencies")
print(f"  - tier_3_agencies: {len(tier_3_agencies)} agencies")

In [ ]:
# Set random seed for reproducibility
RANDOM_STATE = 42

# ==============================================================================
# TIER 1: TARGETED SAMPLING FROM PROBLEMATIC AGENCIES

print("=" * 100)
print("EXECUTING TIER 1: TARGETED SAMPLING")
print("=" * 100)

tier_1_samples = []

# Sample from each Tier 1 agency according to allocation plan
for _, row in tier_1_df.iterrows():
    agency = row["agency"]
    n_to_sample = row["actual_sample"]

    # Get all cases from this agency
    agency_cases = df[df["source_agency_name"] == agency]["provisional_case_name"].unique()

    # Randomly sample the required number of cases
    sampled_cases = np.random.choice(agency_cases, size=n_to_sample, replace=False)

    # Get all rows for these sampled cases
    agency_sample = df[
        (df["source_agency_name"] == agency) & (df["provisional_case_name"].isin(sampled_cases))
    ].copy()

    # Add tier label
    agency_sample["sampling_tier"] = "tier_1_targeted"
    agency_sample["tier_description"] = "Problematic agencies (targeted)"

    tier_1_samples.append(agency_sample)

    print(f"  ✓ Sampled {n_to_sample} cases from {agency}")

# Combine all Tier 1 samples
tier_1_sample_df = pd.concat(tier_1_samples, ignore_index=True)

print(
    f"\nTier 1 complete: {tier_1_sample_df['provisional_case_name'].nunique()} unique cases sampled"
)
print(f"Total rows (including all documents per case): {len(tier_1_sample_df)}")

# ==============================================================================
# TIER 2: RANDOM SAMPLING FROM MEDICAL EXAMINER & DA AGENCIES

print("\n" + "=" * 100)
print("EXECUTING TIER 2: RANDOM SAMPLING FROM MEDICAL/DA AGENCIES")
print("=" * 100)

# Get all cases from Tier 2 agencies
tier_2_cases = df[df["source_agency_name"].isin(tier_2_agencies)]["provisional_case_name"].unique()

# Randomly sample the required number of cases
tier_2_sampled_cases = np.random.choice(tier_2_cases, size=tier_2_allocation, replace=False)

# Get all rows for these sampled cases
tier_2_sample_df = df[
    (df["source_agency_name"].isin(tier_2_agencies))
    & (df["provisional_case_name"].isin(tier_2_sampled_cases))
].copy()

# Add tier label
tier_2_sample_df["sampling_tier"] = "tier_2_random"
tier_2_sample_df["tier_description"] = "Medical examiner & DA agencies (random)"

print(
    f"Tier 2 complete: {tier_2_sample_df['provisional_case_name'].nunique()} unique cases sampled"
)
print(f"Total rows (including all documents per case): {len(tier_2_sample_df)}")

# Show agency breakdown
tier_2_agency_counts = tier_2_sample_df.groupby("source_agency_name")[
    "provisional_case_name"
].nunique()
print(f"\nSampled from {len(tier_2_agency_counts)} different agencies:")
print(tier_2_agency_counts.head(10))

# ==============================================================================
# TIER 3: RANDOM SAMPLING FROM ALL OTHER AGENCIES

print("\n" + "=" * 100)
print("EXECUTING TIER 3: RANDOM SAMPLING FROM ALL OTHER AGENCIES")
print("=" * 100)

# Get all cases from Tier 3 agencies
tier_3_cases = df[df["source_agency_name"].isin(tier_3_agencies)]["provisional_case_name"].unique()

# Randomly sample the required number of cases
tier_3_sampled_cases = np.random.choice(tier_3_cases, size=tier_3_allocation, replace=False)

# Get all rows for these sampled cases
tier_3_sample_df = df[
    (df["source_agency_name"].isin(tier_3_agencies))
    & (df["provisional_case_name"].isin(tier_3_sampled_cases))
].copy()

# Add tier label
tier_3_sample_df["sampling_tier"] = "tier_3_random"
tier_3_sample_df["tier_description"] = "All other agencies (random)"

print(
    f"Tier 3 complete: {tier_3_sample_df['provisional_case_name'].nunique()} unique cases sampled"
)
print(f"Total rows (including all documents per case): {len(tier_3_sample_df)}")

# Show agency breakdown
tier_3_agency_counts = tier_3_sample_df.groupby("source_agency_name")[
    "provisional_case_name"
].nunique()
print(f"\nSampled from {len(tier_3_agency_counts)} different agencies:")
print(tier_3_agency_counts.head(10))

print("\n" + "=" * 100)
print("COMBINING ALL TIERS INTO FINAL SAMPLE")
print("=" * 100)

# Concatenate all three tiers
final_sample = pd.concat([tier_1_sample_df, tier_2_sample_df, tier_3_sample_df], ignore_index=True)

# Verify sample composition
print("\nFinal sample composition:")
print(f"  - Total unique cases: {final_sample['provisional_case_name'].nunique()}")
print(f"  - Total rows (all documents): {len(final_sample)}")
print("\nBreakdown by tier:")
tier_breakdown = final_sample.groupby("sampling_tier")["provisional_case_name"].nunique()
print(tier_breakdown)

total_cases = final_sample["provisional_case_name"].nunique()

expected_cases = tier_1_total + tier_2_allocation + tier_3_allocation


if total_cases == expected_cases:
    print(f"\n✓ Sample verification passed: {total_cases} unique cases = {expected_cases} expected")
else:
    print(f"\n⚠️  WARNING: Found {total_cases} unique cases but expected {expected_cases}")

print("\n" + "=" * 100)
print("SAVING FINAL SAMPLE")
print("=" * 100)

output_path = "../data/output/sample_1000_three_tier.csv"
final_sample.to_csv(output_path, index=False)
print(f"✓ Saved final sample to: {output_path}")

print("\n" + "=" * 100)
print("SUMMARY STATISTICS")
print("=" * 100)

print("\nTop 10 agencies by case count in final sample:")
agency_distribution = (
    final_sample.groupby("source_agency_name")["provisional_case_name"]
    .nunique()
    .sort_values(ascending=False)
)
print(agency_distribution.head(10))

print("\nCases by sampling tier:")
tier_summary = final_sample.groupby(["sampling_tier", "tier_description"])[
    "provisional_case_name"
].nunique()
print(tier_summary)